In [ ]:
# Kaggle P100-compatible stack (sm_60)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!pip uninstall -y -q torch torchvision torchaudio
!pip install -q --index-url https://download.pytorch.org/whl/cu118 torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1
!pip install -q transformers==4.46.3 datasets evaluate accelerate rouge-score bert-score


In [ ]:
# Qwen EncDec Bridge Model
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.modeling_outputs import Seq2SeqLMOutput


@dataclass
class QwenEncDecBridgeConfig:
    encoder_model: str
    decoder_model: Optional[str] = None
    output_dir: str = ""
    max_source_length: int = 1024
    max_target_length: int = 256
    tied_embeddings: bool = True
    bridge_layers: int = 1
    share_backbone_weights: bool = True

    def __post_init__(self):
        if self.decoder_model is None:
            self.decoder_model = self.encoder_model


class CrossAttentionBridge(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, num_layers: int = 1):
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.MultiheadAttention(hidden_size, num_heads, batch_first=True) for _ in range(num_layers)]
        )
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_size) for _ in range(num_layers)])

    def forward(self, query_states: torch.Tensor, encoder_states: torch.Tensor, encoder_mask: Optional[torch.Tensor] = None):
        key_padding_mask = None
        if encoder_mask is not None:
            key_padding_mask = encoder_mask == 0

        x = query_states
        for attn, norm in zip(self.layers, self.norms):
            attn_out, _ = attn(x, encoder_states, encoder_states, key_padding_mask=key_padding_mask, need_weights=False)
            x = norm(x + attn_out)
        return x


class QwenEncDecBridgeModel(nn.Module):
    def __init__(self, encoder_lm: nn.Module, decoder_lm: nn.Module, pad_token_id: int, eos_token_id: int, bridge_layers: int = 1):
        super().__init__()
        self.encoder_lm = encoder_lm
        self.decoder_lm = decoder_lm

        self.encoder = getattr(encoder_lm, "model", encoder_lm)
        self.decoder = getattr(decoder_lm, "model", decoder_lm)

        hidden_size = self.decoder.config.hidden_size
        num_heads = self.decoder.config.num_attention_heads
        self.bridge = CrossAttentionBridge(hidden_size=hidden_size, num_heads=num_heads, num_layers=bridge_layers)

        self.pad_token_id = pad_token_id
        self.eos_token_id = eos_token_id

        self.config = self.decoder_lm.config
        self.config.pad_token_id = pad_token_id
        self.config.eos_token_id = eos_token_id
        self._gc_enabled = False

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        self._gc_enabled = True
        if hasattr(self.encoder_lm, "gradient_checkpointing_enable"):
            try:
                self.encoder_lm.gradient_checkpointing_enable(
                    gradient_checkpointing_kwargs=gradient_checkpointing_kwargs
                )
            except TypeError:
                self.encoder_lm.gradient_checkpointing_enable()
        # Avoid double-call if encoder and decoder share the same object.
        if self.decoder_lm is not self.encoder_lm and hasattr(self.decoder_lm, "gradient_checkpointing_enable"):
            try:
                self.decoder_lm.gradient_checkpointing_enable(
                    gradient_checkpointing_kwargs=gradient_checkpointing_kwargs
                )
            except TypeError:
                self.decoder_lm.gradient_checkpointing_enable()

    def gradient_checkpointing_disable(self):
        self._gc_enabled = False
        if hasattr(self.encoder_lm, "gradient_checkpointing_disable"):
            self.encoder_lm.gradient_checkpointing_disable()
        if self.decoder_lm is not self.encoder_lm and hasattr(self.decoder_lm, "gradient_checkpointing_disable"):
            self.decoder_lm.gradient_checkpointing_disable()

    @property
    def is_gradient_checkpointing(self):
        return self._gc_enabled

    def freeze_backbones(self):
        for p in self.encoder.parameters():
            p.requires_grad = False
        for p in self.decoder.parameters():
            p.requires_grad = False
        if hasattr(self.decoder_lm, "lm_head"):
            for p in self.decoder_lm.lm_head.parameters():
                p.requires_grad = False

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True

    def shift_right(self, labels: torch.Tensor):
        shifted = torch.full_like(labels, fill_value=self.pad_token_id)
        shifted[:, 1:] = labels[:, :-1]
        shifted[:, 0] = self.eos_token_id
        shifted = shifted.masked_fill(shifted == -100, self.pad_token_id)
        return shifted

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        decoder_input_ids: Optional[torch.Tensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        **kwargs,
    ):
        enc_out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_hidden_states=False,
            return_dict=True,
        )
        encoder_hidden = enc_out.last_hidden_state

        if decoder_input_ids is None:
            if labels is None:
                raise ValueError("Need either decoder_input_ids or labels")
            decoder_input_ids = self.shift_right(labels)

        if decoder_attention_mask is None:
            decoder_attention_mask = (decoder_input_ids != self.pad_token_id).long()

        dec_embed = self.decoder_lm.get_input_embeddings()(decoder_input_ids)
        bridge_first_param = next(self.bridge.parameters(), None)
        bridge_dtype = bridge_first_param.dtype if bridge_first_param is not None else dec_embed.dtype
        dec_embed_bridge = dec_embed.to(bridge_dtype)
        encoder_hidden_bridge = encoder_hidden.to(bridge_dtype)
        bridged_embed = self.bridge(dec_embed_bridge, encoder_hidden_bridge, attention_mask)
        dec_dtype = self.decoder_lm.get_input_embeddings().weight.dtype
        bridged_embed = bridged_embed.to(dec_dtype)

        dec_out = self.decoder_lm(
            inputs_embeds=bridged_embed,
            attention_mask=decoder_attention_mask,
            labels=labels,
            use_cache=False,
            return_dict=True,
        )

        return Seq2SeqLMOutput(
            loss=dec_out.loss,
            logits=dec_out.logits,
            encoder_last_hidden_state=encoder_hidden,
        )

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        max_new_tokens: int = 128,
        num_beams: int = 1,
    ):
        if num_beams != 1:
            raise ValueError("This lightweight bridge currently supports greedy decoding only (num_beams=1).")

        bsz = input_ids.size(0)
        device = input_ids.device
        generated = torch.full((bsz, 1), self.eos_token_id, dtype=torch.long, device=device)

        for _ in range(max_new_tokens):
            out = self.forward(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=generated,
            )
            next_token_logits = out.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if torch.all(next_token.squeeze(-1) == self.eos_token_id):
                break

        return generated[:, 1:]


def build_qwen_encdec_bridge(cfg: QwenEncDecBridgeConfig):
    tokenizer = AutoTokenizer.from_pretrained(cfg.encoder_model, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    encoder_lm = AutoModelForCausalLM.from_pretrained(cfg.encoder_model)
    if cfg.decoder_model == cfg.encoder_model and cfg.share_backbone_weights:
        decoder_lm = encoder_lm
    else:
        decoder_lm = AutoModelForCausalLM.from_pretrained(cfg.decoder_model)

    encoder_lm = encoder_lm.float()
    if decoder_lm is not encoder_lm:
        decoder_lm = decoder_lm.float()

    model = QwenEncDecBridgeModel(
        encoder_lm=encoder_lm,
        decoder_lm=decoder_lm,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        bridge_layers=cfg.bridge_layers,
    )

    if cfg.tied_embeddings:
        model.encoder.set_input_embeddings(model.decoder_lm.get_input_embeddings())

    return model, tokenizer



In [ ]:
# Warmup Callback
import torch
from transformers import TrainerCallback, TrainerState, TrainerControl

class FreezeNonCrossAttentionCallback(TrainerCallback):
    """
    A callback that freezes all parameters except cross-attention layers during a warmup phase.
    """
    def __init__(self, warmup_steps: int):
        self.warmup_steps = warmup_steps
        self.is_frozen = False

    def on_train_begin(self, args, state: TrainerState, control: TrainerControl, model=None, **kwargs):
        if self.warmup_steps > 0 and model is not None:
            print(f"Starting cross-attention warmup for {self.warmup_steps} steps. Freezing non-cross-attention parameters.")
            self._freeze_non_cross_attention(model)
            self.is_frozen = True

    def on_step_end(self, args, state: TrainerState, control: TrainerControl, model=None, **kwargs):
        if self.is_frozen and state.global_step >= self.warmup_steps and model is not None:
            print(f"Warmup finished at step {state.global_step}. Unfreezing all parameters.")
            self._unfreeze_all(model)
            self.is_frozen = False

    def _freeze_non_cross_attention(self, model):
        # By default freeze everything
        for param in model.parameters():
            param.requires_grad = False
            
        # Unfreeze cross-attention specifically
        for name, param in model.named_parameters():
            lname = name.lower()
            if "crossattention" in lname or "encoder_attn" in lname or "bridge" in lname:
                param.requires_grad = True
                
    def _unfreeze_all(self, model):
        for param in model.parameters():
            param.requires_grad = True



In [ ]:
# Data Utils
from typing import Dict, Any
from datasets import load_dataset
from transformers import AutoTokenizer

def load_and_preprocess_dataset(
    dataset_name: str,
    tokenizer: AutoTokenizer,
    text_column: str,
    summary_column: str,
    prefix: str = "",
    max_source_length: int = 2048,
    max_target_length: int = 512,
    train_samples: int = -1,
    eval_samples: int = -1,
    dataset_config: str = None
):
    """
    Loads dataset (like nam194/vietnews), applies prefix, and tokenizes.
    """
    if dataset_config:
        ds = load_dataset(dataset_name, dataset_config)
    else:
        ds = load_dataset(dataset_name)

    train_ds = ds["train"]
    if train_samples > 0:
        train_ds = train_ds.select(range(min(train_samples, len(train_ds))))
        
    val_split = "validation" if "validation" in ds else "test"
    eval_ds = ds[val_split]
    if eval_samples > 0:
        eval_ds = eval_ds.select(range(min(eval_samples, len(eval_ds))))

    def preprocess(batch):
        inputs = [prefix + str(doc) for doc in batch[text_column]]
        model_inputs = tokenizer(
            inputs,
            max_length=max_source_length,
            truncation=True,
        )
        labels = tokenizer(
            text_target=batch[summary_column],
            max_length=max_target_length,
            truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
    eval_tok = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)

    return train_tok, eval_tok



In [ ]:
config = {
    "encoder_model": "Qwen/Qwen2.5-0.5B",
    "decoder_model": "Qwen/Qwen2.5-0.5B",
    "share_backbone_weights": True,
    "tied_embeddings": True,
    "bridge_layers": 2,
    "dataset": "nam194/vietnews",
    "text_column": "article",
    "summary_column": "abstract",
    "prefix": "vietnews: ",
    "warmup_steps": 0,
    "freeze_backbones_only": True,
    "max_source_length": 512,
    "max_target_length": 128,
    "train_samples": 4000,
    "eval_samples": 500,
    "epochs": 10,
    "batch_size": 1,
    "grad_accum_steps": 4,
    "lr": 3e-5,
}



In [ ]:
import torch
import numpy as np
import inspect
import evaluate
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook. Please enable GPU in Kaggle settings.")

print("Building Qwen-EncDec Bridge model...")
cfg = QwenEncDecBridgeConfig(
    encoder_model=config["encoder_model"],
    decoder_model=config["decoder_model"],
    share_backbone_weights=config["share_backbone_weights"],
    tied_embeddings=config["tied_embeddings"],
    bridge_layers=config["bridge_layers"],
)
model, tokenizer = build_qwen_encdec_bridge(cfg)
if config.get("freeze_backbones_only", True):
    print("Freezing backbones. Training bridge only for memory efficiency.")
    model.freeze_backbones()
use_gc = config.get("use_gradient_checkpointing", not config.get("freeze_backbones_only", True))
model.config.use_cache = not use_gc
if use_gc:
    if hasattr(model.encoder_lm, "gradient_checkpointing_enable"):
        model.encoder_lm.gradient_checkpointing_enable()
    if model.decoder_lm is not model.encoder_lm and hasattr(model.decoder_lm, "gradient_checkpointing_enable"):
        model.decoder_lm.gradient_checkpointing_enable()
else:
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()

print("Loading dataset...")
train_tok, eval_tok = load_and_preprocess_dataset(
    dataset_name=config["dataset"],
    tokenizer=tokenizer,
    text_column=config["text_column"],
    summary_column=config["summary_column"],
    prefix=config["prefix"],
    max_source_length=config["max_source_length"],
    max_target_length=config["max_target_length"],
    train_samples=config["train_samples"],
    eval_samples=config["eval_samples"],
)

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    if hasattr(preds, "ndim") and preds.ndim == 3:
        preds = preds.argmax(-1)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    scores = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {k: round(v, 4) for k, v in scores.items()}

training_kwargs = dict(
    output_dir="outputs/qwen-bridge-vietnews",
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["batch_size"],
    gradient_accumulation_steps=config["grad_accum_steps"],
    learning_rate=config["lr"],
    num_train_epochs=config["epochs"],
    logging_steps=50,
    eval_strategy="no",
    save_strategy="no",
    predict_with_generate=False,
    fp16=True,
    gradient_checkpointing=use_gc,
    save_safetensors=False,
    report_to="none",
)

sig_args = inspect.signature(Seq2SeqTrainingArguments.__init__)
if "use_cpu" in sig_args.parameters:
    training_kwargs["use_cpu"] = False
elif "no_cuda" in sig_args.parameters:
    training_kwargs["no_cuda"] = False

training_args = Seq2SeqTrainingArguments(**training_kwargs)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=None)

callbacks = []
if config["warmup_steps"] > 0 and not config.get("freeze_backbones_only", True):
    callbacks.append(FreezeNonCrossAttentionCallback(warmup_steps=config["warmup_steps"]))

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)
sig_tr = inspect.signature(Seq2SeqTrainer.__init__)
if "processing_class" in sig_tr.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sig_tr.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)
print("Starting training...")
trainer.train()
torch.cuda.empty_cache()

import os
os.makedirs("outputs/qwen-bridge-vietnews", exist_ok=True)
torch.save(model.state_dict(), "outputs/qwen-bridge-vietnews/pytorch_model.bin")
tokenizer.save_pretrained("outputs/qwen-bridge-vietnews")
print("Training completed and checkpoint saved.")



In [ ]:
import torch
from datasets import load_dataset
import evaluate

print("Starting Evaluation...")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for evaluation. Please enable GPU in Kaggle settings.")
device = "cuda"
model.to(device)
model.eval()

bertscore = evaluate.load("bertscore")

ds = load_dataset(config["dataset"])
split = "test" if "test" in ds else "validation"
subset = ds[split].select(range(min(config["eval_samples"], len(ds[split]))))

preds, refs = [], []
for ex in subset:
    src = config["prefix"] + str(ex[config["text_column"]])
    inputs = tokenizer(src, return_tensors="pt", truncation=True, max_length=config["max_source_length"]).to(device)
    with torch.inference_mode():
        out = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask", None),
            max_new_tokens=config["max_target_length"],
            num_beams=1,
        )
    preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
    refs.append(ex[config["summary_column"]])

rouge_scores = rouge.compute(predictions=preds, references=refs, use_stemmer=False)
bert_scores = bertscore.compute(predictions=preds, references=refs, lang="vi", device="cuda:0")

print("\n[ROUGE Scores]")
for k, v in rouge_scores.items():
    print(f"  {k}: {v:.4f}")

print("\n[BERTScore]")
print(f"  precision: {sum(bert_scores['precision'])/len(bert_scores['precision']):.4f}")
print(f"  recall:    {sum(bert_scores['recall'])/len(bert_scores['recall']):.4f}")
print(f"  f1:        {sum(bert_scores['f1'])/len(bert_scores['f1']):.4f}")

